# 03 Locators and progress

**Graduates to:** `hub/src/hudhud_hub/progress.py`

## What is a locator?

"Page 63" means nothing in a reflowable book: change the font size and page
63 holds different words. So readers store a **locator** that points at the
text itself.

- **EPUB CFI** (Canonical Fragment Identifier) is a path through the book,
  for example `epubcfi(/6/4!/4/2/1:3)`. Read it left to right: `/6` is the
  spine in the package document, `/4` is the 2nd item in the spine (even
  numbers are elements, so `/4` means element 2), `!` steps into that
  chapter's XHTML, `/4/2/1` walks down into the body, and `:3` is a character
  offset. foliate-js produces CFIs for every format it renders, so the hub
  can treat them as opaque strings. **The hub never parses a CFI; it stores
  and returns it.**
- **PDF** pages do not reflow, so the locator is `{page, fraction}`.
- Every locator also carries a `fraction` from 0 to 1. It drives the progress
  bar and is the fallback when a precise locator no longer resolves.

## Progress is per device

The phone and the PC each have their own row: primary key
`(book_id, device_id)`. Nothing ever has to merge. A row only moves forward
in time: a write with an older `updated_at` than the stored one is ignored,
so a slow retry from an hour ago cannot drag you back.

SQLite does this in one statement:

```sql
INSERT ... ON CONFLICT (book_id, device_id) DO UPDATE SET ...
WHERE excluded.updated_at > progress.updated_at
```

`excluded` is the row you tried to insert. The ISO timestamps compare
correctly as strings because they all share one fixed format.

## The resume rule

When a book opens, the client compares its own position with the newest
position from any *other* device. It offers "Continue from Phone at 63%?" only
when both of these hold:

- the other position is newer than this device's, and
- the two differ by more than one page (`1 / total_pages` as a fraction).

If this device has never opened the book, treat its position as 0.

## Validate before you compare

String comparison only works if every timestamp has exactly the same shape.
`"...10:00:00.500Z"` sorts *before* `"...10:00:00Z"` (because `.` < `Z`), and
a junk value like `"zzz"` would beat every real time forever. So the hub
rejects any `updated_at` that is not `YYYY-MM-DDTHH:MM:SS.mmmZ`.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
import math
import os
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path

import ipytest

ipytest.autoconfig(raise_on_error=os.environ.get("HUDHUD_NB_CHECK") == "1")

TIMESTAMP_RE = re.compile(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\d\.\d{3}Z")


@dataclass(frozen=True)
class Progress:
    book_id: str
    device_id: str
    locator: str
    fraction: float
    updated_at: str

## Your code

In [ ]:
def upsert_progress(conn: sqlite3.Connection, p: Progress) -> bool:
    """Store p unless the stored row for (book_id, device_id) is as new or newer.

    Raise ValueError if fraction is not within [0, 1] (NaN included), the
    locator is empty, or updated_at does not fully match TIMESTAMP_RE.
    Return True if the row was written.
    """
    raise NotImplementedError


def get_progress(conn: sqlite3.Connection, book_id: str, device_id: str) -> Progress | None:
    """This device's position in this book, or None."""
    raise NotImplementedError


def latest_other_device(conn: sqlite3.Connection, book_id: str, device_id: str) -> Progress | None:
    """The newest position for this book from any device except `device_id`."""
    raise NotImplementedError


def should_offer_resume(own: Progress | None, other: Progress | None, total_pages: int) -> bool:
    """Apply the resume rule above. ValueError if total_pages < 1."""
    raise NotImplementedError

## Tests

In [ ]:
SCHEMA = Path("../hub/src/hudhud_hub/migrations/0001_init.sql").read_text(encoding="utf-8")
BOOK = "b" * 64


def db():
    conn = sqlite3.connect(":memory:")
    conn.executescript(SCHEMA)
    conn.execute(
        "INSERT INTO books (id, title, format, file_size, added_at, updated_at) "
        "VALUES (?, 'Divan', 'epub', 1, 't', 't')", (BOOK,))
    return conn


def p(device="pc", fraction=0.5, at="2026-10-08T10:00:00.000Z", locator="epubcfi(/6/4!/4/2/1:3)"):
    return Progress(BOOK, device, locator, fraction, at)

In [ ]:
%%ipytest -q
import pytest


def test_first_write_is_stored():
    conn = db()
    assert upsert_progress(conn, p()) is True
    assert get_progress(conn, BOOK, "pc") == p()


def test_newer_wins_older_is_ignored():
    conn = db()
    upsert_progress(conn, p(fraction=0.5, at="2026-10-08T10:00:00.000Z"))
    assert upsert_progress(conn, p(fraction=0.6, at="2026-10-08T11:00:00.000Z")) is True
    assert upsert_progress(conn, p(fraction=0.1, at="2026-10-08T09:00:00.000Z")) is False
    assert get_progress(conn, BOOK, "pc").fraction == 0.6


def test_equal_timestamp_is_ignored():
    conn = db()
    upsert_progress(conn, p(fraction=0.5))
    assert upsert_progress(conn, p(fraction=0.9)) is False
    assert get_progress(conn, BOOK, "pc").fraction == 0.5


@pytest.mark.parametrize("fraction", [-0.01, 1.01, math.nan])
def test_bad_fraction_rejected(fraction):
    with pytest.raises(ValueError):
        upsert_progress(db(), p(fraction=fraction))


@pytest.mark.parametrize("at", ["2026-10-08T10:00:00Z", "2026-10-08 10:00:00.000Z", "zzz", ""])
def test_malformed_timestamp_rejected(at):
    with pytest.raises(ValueError):
        upsert_progress(db(), p(at=at))


def test_empty_locator_rejected():
    with pytest.raises(ValueError):
        upsert_progress(db(), p(locator=""))


def test_locator_is_stored_verbatim():
    conn = db()
    weird = "epubcfi(/6/14!/4/2[فصل-۱]/1:0)"
    upsert_progress(conn, p(locator=weird))
    assert get_progress(conn, BOOK, "pc").locator == weird


def test_unknown_device_has_no_progress():
    assert get_progress(db(), BOOK, "phone") is None


def test_latest_other_device_skips_own_and_picks_newest():
    conn = db()
    upsert_progress(conn, p("pc", 0.9, "2026-10-08T12:00:00.000Z"))
    upsert_progress(conn, p("phone", 0.3, "2026-10-08T10:00:00.000Z"))
    upsert_progress(conn, p("tablet", 0.4, "2026-10-08T11:00:00.000Z"))
    assert latest_other_device(conn, BOOK, "pc").device_id == "tablet"
    assert latest_other_device(conn, BOOK, "tablet").device_id == "pc"


def test_no_other_device():
    conn = db()
    upsert_progress(conn, p("pc"))
    assert latest_other_device(conn, BOOK, "pc") is None


def test_resume_rule():
    early, late = "2026-10-08T10:00:00.000Z", "2026-10-08T11:00:00.000Z"
    assert should_offer_resume(None, None, 300) is False
    assert should_offer_resume(None, p("phone", 0.63, late), 300) is True
    assert should_offer_resume(p("pc", 0.20, early), p("phone", 0.63, late), 300) is True
    assert should_offer_resume(p("pc", 0.500, early), p("phone", 0.502, late), 300) is False
    assert should_offer_resume(p("pc", 0.20, late), p("phone", 0.63, early), 300) is False
    assert should_offer_resume(p("pc", 0.63, early), p("phone", 0.20, late), 300) is True


def test_resume_rule_needs_pages():
    with pytest.raises(ValueError):
        should_offer_resume(None, p(), 0)

## Graduate

1. `hub/src/hudhud_hub/progress.py`: imports (minus `ipytest`, `Path`),
   `Progress` and your four functions.
2. `hub/tests/test_progress.py`: the fixture and test cells, with
   `from hudhud_hub.progress import *` and `SCHEMA` read from the package's
   migrations folder.
3. `uv run --project hub pytest hub/tests/test_progress.py -q`